# Statcast Snapshot Profiling
Profiles the frozen snapshot to set cleaning thresholds and resolve two open questions about position player pitchers and non competitive pitches. This notebook only measures the data and applies no cleaning, and its results feed the rules built in src/data/clean.py.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration
Snapshot location and the field groups profiled below. Bat tracking fields are retained for later phases but excluded from the version one feature vector.

In [2]:
pd.set_option("display.max_rows", 120)

SNAPSHOT_DIR = Path("../data/raw/statcast/snapshot_2026-07-14")
SEASON_FILES = sorted(SNAPSHOT_DIR.glob("season=*.parquet"))

# a player with at least this many batting appearances in a season counts as primarily a batter
BATTER_PA_THRESHOLD = 50
# release speed above which a batter who also pitched is treated as a real or two way pitcher
VELOCITY_SEPARATION = 80.0

KEY_FIELDS = ["game_pk", "at_bat_number", "pitch_number"]
ESSENTIAL_FIELDS = ["description", "balls", "strikes", "p_throws", "stand", "pitch_type"]
CONTEXT_FIELDS = [
    "release_speed", "effective_speed", "release_spin_rate", "spin_axis",
    "pfx_x", "pfx_z", "plate_x", "plate_z",
    "release_pos_x", "release_pos_y", "release_pos_z", "release_extension", "zone",
]
CONTACT_QUALITY_FIELDS = ["launch_speed", "launch_angle", "hc_x", "hc_y", "hit_distance_sc"]
BAT_TRACKING_FIELDS = ["bat_speed", "swing_length", "attack_angle", "attack_direction", "swing_path_tilt"]

PROFILE_FIELDS = ESSENTIAL_FIELDS + CONTEXT_FIELDS + CONTACT_QUALITY_FIELDS + BAT_TRACKING_FIELDS
RANGE_FIELDS = ["release_speed", "effective_speed", "plate_x", "plate_z", "launch_speed", "launch_angle", "release_spin_rate"]
COLUMNS_TO_LOAD = sorted(set(KEY_FIELDS + PROFILE_FIELDS + ["game_type", "batter", "pitcher"]))

## Load regular season pitches
Read every season file from the snapshot, tag each with its season, and keep the regular season rows the model trains on. The full pull is held separately so the game type composition can still be reported.

In [3]:
season_frames = []
for path in SEASON_FILES:
    season = int(path.stem.split("=")[1])
    frame = pd.read_parquet(path, columns=COLUMNS_TO_LOAD)
    frame["season"] = season
    season_frames.append(frame)
all_pitches = pd.concat(season_frames, ignore_index=True)

regular_season = all_pitches[all_pitches["game_type"] == "R"]
print(f"loaded pitches: {len(all_pitches)}")
print(f"regular season pitches: {len(regular_season)}")

loaded pitches: 7799380
regular season pitches: 7434200


## Game type composition
Show how many pitches per season are regular season versus spring training and postseason. This confirms how much the regular season filter removes.

In [4]:
game_type_table = all_pitches.groupby("season")["game_type"].value_counts().unstack(fill_value=0)
game_type_table["total"] = game_type_table.sum(axis=1)
game_type_table["regular_share"] = (game_type_table["R"] / game_type_table["total"]).round(3)
game_type_table

game_type,D,F,L,R,S,W,total,regular_share
season,,,,,,,,
2015,5552,524,2844,702306,0,1618,712844,0.985
2016,4497,571,2995,715823,252,2137,726275,0.986
2017,5287,655,3335,724627,0,2050,735954,0.985
2018,4147,666,3637,724452,0,1665,734567,0.986
2019,5477,560,2977,735090,16926,2168,763198,0.963
2020,4517,5523,4330,264262,0,1766,280398,0.942
2021,4979,574,3557,712320,42561,1742,765733,0.930
2022,4796,2743,2509,710210,53346,1726,775330,0.916
2023,4126,2291,3964,720684,41476,1497,774038,0.931


## Field missingness
Report the fraction of regular season pitches missing each candidate field, by season. Fields fully missing in early seasons are structural and are kept with missingness indicators rather than dropped.

In [5]:
missingness_fraction = regular_season.groupby("season")[PROFILE_FIELDS].apply(lambda group: group.isna().mean()).round(3)
missingness_fraction.T

season,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
description,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
balls,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
strikes,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
p_throws,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
stand,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
pitch_type,0.001,0.001,0.009,0.009,0.010,0.003,0.004,0.003,0.004,0.004,0.004
release_speed,0.000,0.000,0.008,0.009,0.010,0.003,0.004,0.003,0.004,0.004,0.004
effective_speed,0.021,0.015,0.012,0.009,0.010,0.005,0.005,0.005,0.006,0.005,0.004
release_spin_rate,0.071,0.079,0.028,0.024,0.027,0.007,0.008,0.006,0.010,0.008,0.006
spin_axis,1.000,0.015,0.008,0.009,0.010,0.007,0.008,0.006,0.010,0.008,0.006


## Duplicate pitch keys
Count rows that share a game_pk, at_bat_number, and pitch_number key. Any duplicates found here are removed during cleaning.

In [6]:
duplicate_flag = regular_season.duplicated(subset=KEY_FIELDS)
duplicate_counts = duplicate_flag.groupby(regular_season["season"]).sum()
duplicate_counts.rename("duplicate_keys").to_frame()

,duplicate_keys
season,
2015,0
2016,0
2017,0
2018,0
2019,0
2020,0
2021,0
2022,0
2023,0


## Physical range extremes
Show the minimum, 1st percentile, 99th percentile, and maximum of each numeric field. A wide gap between the 99th percentile and the maximum flags impossible readings that need hard bounds.

In [7]:
range_summary = pd.DataFrame({
    "min": regular_season[RANGE_FIELDS].min(),
    "p01": regular_season[RANGE_FIELDS].quantile(0.01),
    "p99": regular_season[RANGE_FIELDS].quantile(0.99),
    "max": regular_season[RANGE_FIELDS].max(),
})
range_summary

,min,p01,p99,max
release_speed,21.7,73.3,98.8,105.7
effective_speed,26.4,72.3,99.3,107.1
plate_x,-11.365394,-1.958915,2.019693,34.972497
plate_z,-57.60148,-0.06425,4.548905,13.579718
launch_speed,2.0,40.0,109.9,123.9
launch_angle,-90.0,-59.0,78.0,90.0
release_spin_rate,14.0,1137.0,2974.0,3741.0


## Pitch type frequencies
List how often each pitch type code appears in the regular season. This exposes the codes for intentional balls and pitchouts so the label logic uses real codes rather than assumed ones.

In [8]:
regular_season["pitch_type"].value_counts(dropna=False)

pitch_type
FF     2534891
SI     1292808
SL     1140601
CH      783939
CU      548190
FC      500999
ST      224225
KC      170600
FS      144613
NaN      35704
SV       24192
KN       13087
IN        6200
FA        5588
EP        3657
FO        2523
CS        1053
PO         937
SC         348
UN          42
AB           3
Name: count, dtype: int64

## Description frequencies
List how often each pitch description appears. This reveals non competitive events such as automatic balls and pitchouts that may need exclusion from the swing decision factor.

In [9]:
regular_season["description"].value_counts()

description
ball                       2481328
foul                       1309579
hit_into_play              1302244
called_strike              1228777
swinging_strike             752226
blocked_ball                169723
foul_tip                     68424
swinging_strike_blocked      52197
automatic_ball               21119
hit_by_pitch                 19961
foul_bunt                    16780
intent_ball                   6602
missed_bunt                   3437
pitchout                       927
automatic_strike               536
bunt_foul_tip                  331
swinging_pitchout                6
foul_pitchout                    3
Name: count, dtype: int64

## Position player pitcher candidates
Summarize pitches thrown by players who are primarily batters, using the batting appearance threshold. The median release speed exposes a confound from the pre 2022 no designated hitter era, when real pitchers still batted.

In [10]:
position_player_rows = []
for season, frame in regular_season.groupby("season"):
    batting_appearances = frame[["batter", "game_pk", "at_bat_number"]].drop_duplicates().groupby("batter").size()
    primarily_batters = set(batting_appearances[batting_appearances >= BATTER_PA_THRESHOLD].index)
    position_player_pitches = frame[frame["pitcher"].isin(primarily_batters)]
    position_player_rows.append({
        "season": season,
        "pitches": len(position_player_pitches),
        "pitcher_ids": position_player_pitches["pitcher"].nunique(),
        "max_release_speed": position_player_pitches["release_speed"].max(),
        "median_release_speed": position_player_pitches["release_speed"].median(),
    })
position_player_table = pd.DataFrame(position_player_rows).set_index("season")
position_player_table

,pitches,pitcher_ids,max_release_speed,median_release_speed
season,,,,
2015,120810,61,101.0,90.30
2016,112652,59,102.1,90.30
2017,116649,61,100.9,89.80
2018,112259,85,101.1,89.70
2019,119996,95,100.6,89.60
2020,339,17,97.1,73.25
2021,83486,85,101.5,89.40
2022,4543,62,101.4,84.20
2023,3845,55,101.2,82.30


## Position player pitcher separation
For players who batted often and also pitched, split them by a velocity threshold and show how many batters each group faced. Real pitchers who batted before 2022 throw hard and face many batters, while position players throw soft and face very few, which is the separation the cleaning rule relies on.

In [11]:
separation_rows = []
for season, frame in regular_season.groupby("season"):
    batting_appearances = frame[["batter", "game_pk", "at_bat_number"]].drop_duplicates().groupby("batter").size()
    batters_faced = frame[["pitcher", "game_pk", "at_bat_number"]].drop_duplicates().groupby("pitcher").size()
    max_release_speed = frame.groupby("pitcher")["release_speed"].max()

    # players who batted often and also threw at least one pitch this season
    frequent_batters = batting_appearances[batting_appearances >= BATTER_PA_THRESHOLD].index
    also_pitched = [player for player in frequent_batters if player in batters_faced.index]
    flagged = pd.DataFrame({"batters_faced": batters_faced.reindex(also_pitched),
                            "max_release_speed": max_release_speed.reindex(also_pitched)})

    hard_throwers = flagged[flagged["max_release_speed"] >= VELOCITY_SEPARATION]
    soft_throwers = flagged[flagged["max_release_speed"] < VELOCITY_SEPARATION]
    separation_rows.append({
        "season": season,
        "hard_throwers": len(hard_throwers),
        "hard_median_batters_faced": hard_throwers["batters_faced"].median(),
        "soft_throwers": len(soft_throwers),
        "soft_median_batters_faced": soft_throwers["batters_faced"].median(),
    })
separation_table = pd.DataFrame(separation_rows).set_index("season")
separation_table

,hard_throwers,hard_median_batters_faced,soft_throwers,soft_median_batters_faced
season,,,,
2015,58,753.5,3,6.0
2016,58,699.0,1,3.0
2017,54,702.5,7,6.0
2018,71,627.0,13,3.0
2019,77,624.0,17,7.0
2020,6,7.5,11,3.0
2021,58,564.5,27,5.0
2022,28,10.0,34,7.0
2023,21,10.0,34,10.5


## Decisions this profiling feeds
The cleaning rules the tables above inform before src/data/clean.py is written.

- Hard physical bounds for the location, velocity, and spin fields from the range table.
- A position player pitcher rule that combines batting role with a velocity cutoff, since batting appearances alone are confounded by the no designated hitter era before 2022.
- Whether to drop intentional balls, pitchouts, and automatic balls from the swing decision factor.
- Confirmation that bat tracking missingness before 2023 is structural and safe to keep with indicators.